In [13]:
"""
clean_data.py  --  Transform step (Milestone 1)

Reads the nested raw REST Countries JSON, flattens it, applies documented
cleaning rules, and writes a clean CSV + a cleaning log.

Usage:
    python src/clean_data.py [raw_json] [out_dir]
Defaults: data/raw/raw_countries_.json  ->  data/clean/
"""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path(sys.argv[1] if len(sys.argv) > 1 else "data/raw/raw_countries_.json")
OUT = Path(sys.argv[2] if len(sys.argv) > 2 else "data/clean")
OUT.mkdir(parents=True, exist_ok=True)

log = []  # (rule, rows_affected, note)


def record(rule, n, note=""):
    log.append({"rule": rule, "rows_affected": int(n), "note": note})


# ---------------------------------------------------------------- 1. load
with open('raw_countries_.json', encoding="utf-8") as f:
    records = json.load(f)
n_raw = len(records)

FLAT_INPUT = bool(records) and "names" not in records[0]
if FLAT_INPUT:
    if "names_common" not in records[0]:
        sys.exit("ERROR: unrecognised input format (expected nested API JSON or the flattened table).")
    print("WARNING: input is the FLATTENED file. language names and currency codes are not "
          "available in it, so those two columns will be empty. Use the nested raw JSON for the full dataset.")

    def _split(v):
        return [x for x in (v or "").split(", ") if x]

    records = [{
        "names": {"common": r.get("names_common")},
        "codes": {"alpha_3": r.get("codes_alpha_3")},
        "region": r.get("region"),
        "population": r.get("population"),
        "area": {"kilometers": r.get("area_kilometers")},
        "borders": _split(r.get("borders")),
        "languages": [{"bcp47": c, "name": None} for c in _split(r.get("languages"))],
        "currencies": [{"name": n, "code": None} for n in _split(r.get("currencies"))],
    } for r in records]

# ------------------------------------------------------------- 2. flatten
rows = []
for r in records:
    langs = r.get("languages") or []
    curs = r.get("currencies") or []
    borders = r.get("borders") or []
    rows.append({
        "name": (r.get("names") or {}).get("common"),
        "iso3": (r.get("codes") or {}).get("alpha_3"),
        "region": r.get("region"),
        "population": r.get("population"),
        "area_km2": (r.get("area") or {}).get("kilometers"),
        "languages": ", ".join(l["name"] for l in langs if l.get("name")),
        "language_codes": ", ".join(l["bcp47"] for l in langs),
        "currencies": ", ".join(c["name"] for c in curs),
        "currency_codes": ", ".join(c["code"] for c in curs if c.get("code")),
        "borders": ", ".join(borders),
        "_lang_codes": [l["bcp47"] for l in langs],
        "_n_borders": len(borders),
        "_n_langs": len(langs),
        "_n_curs": len(curs),
    })
df = pd.DataFrame(rows)
record("flatten nested JSON", n_raw, "languages/currencies/borders -> comma-separated strings + counts")

# ------------------------------------------------ 3. text hygiene / types
for c in ["name", "iso3", "region", "languages", "language_codes",
          "currencies", "currency_codes", "borders"]:
    df[c] = df[c].astype("string").str.strip()

# empty strings are missing values, not real values
for c in ["iso3", "languages", "language_codes", "currencies", "currency_codes"]:
    n = (df[c] == "").sum()
    df[c] = df[c].replace("", pd.NA)
    record(f"'' -> NaN in {c}", n)

df["population"] = pd.to_numeric(df["population"], errors="coerce").astype("Int64")
df["area_km2"] = pd.to_numeric(df["area_km2"], errors="coerce").astype(float)

# ------------------------------------------------------------ 4. dedupe
dups = df.duplicated(subset=["name"]).sum()
df = df.drop_duplicates(subset=["name"])
record("drop duplicate country names", dups)

# ----------------------------------------------------- 5. flag columns
# ISO status: unrecognised states have no alpha-3; Kosovo uses the
# user-assigned code UNK, which is not an official ISO 3166-1 code.
df["is_iso_recognized"] = df["iso3"].notna() & (df["iso3"] != "UNK")
record("flag is_iso_recognized=False", (~df["is_iso_recognized"]).sum(),
       "no alpha-3 (4 rows) or user-assigned code UNK (Kosovo); kept, not dropped")

df["is_uninhabited"] = df["population"].fillna(-1).eq(0)
record("flag is_uninhabited", df["is_uninhabited"].sum(), "population == 0 is a true zero")

df["has_land_border"] = df["_n_borders"] > 0
df["is_antarctic_region"] = df["region"].eq("Antarctic")

# -------------------------------------------------- 6. count features
df["num_borders"] = df["_n_borders"]       # 0 = no land border (NOT missing)
df["num_languages"] = df["_n_langs"]
df["num_currencies"] = df["_n_curs"]
# sign languages / language families are not spoken languages; flag only
SIGN_OR_FAMILY = {"nzs", "zib", "khi"}
df["has_sign_or_family_code"] = df["_lang_codes"].apply(lambda cs: bool(SIGN_OR_FAMILY & set(cs)))
record("flag has_sign_or_family_code", df["has_sign_or_family_code"].sum(),
       "nzs / zib / khi are not spoken languages; counts left unchanged")

# ------------------------------------------------------ 7. derived metrics
valid_density = (df["population"].fillna(0) > 0) & (df["area_km2"] > 0)
df["population_density"] = np.where(
    valid_density, df["population"].astype(float) / df["area_km2"], np.nan)
record("population_density = NaN", (~valid_density).sum(),
       "undefined when population is 0 or area missing")

df["log10_population"] = np.where(df["population"].fillna(0) > 0,
                                  np.log10(df["population"].astype(float).clip(lower=1)), np.nan)
df["log10_area"] = np.where(df["area_km2"] > 0, np.log10(df["area_km2"].clip(lower=1e-9)), np.nan)

# ----------------------------------------------------- 8. outlier flags
# IQR on log10 scale (distributions are heavily right-skewed). Flag only.
def iqr_flag(s):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return lo, hi

for col, lg in [("population", "log10_population"), ("area_km2", "log10_area")]:
    lo, hi = iqr_flag(df[lg])
    flag = f"is_{col}_outlier"
    df[flag] = (df[lg] < lo) | (df[lg] > hi)
    record(f"flag {flag} (IQR on log10)", df[flag].sum(), f"fence = [{lo:.2f}, {hi:.2f}]; kept, not removed")

# --------------------------------------------------------- 9. validation
assert df["name"].is_unique, "duplicate names"
assert df["iso3"].dropna().is_unique, "duplicate ISO3 codes"
assert (df["population"].dropna() >= 0).all(), "negative population"
assert (df["area_km2"].dropna() > 0).all(), "non-positive area"
known = set(df["iso3"].dropna())
bad = {b for s in df["borders"].dropna() for b in s.split(", ") if b and b not in known}
assert not bad, f"border codes missing from dataset: {bad}"
record("validation asserts passed", len(df), "unique names/ISO, no negatives, border refs resolve")

# --------------------------------------------------------------- 10. save
clean = df.drop(columns=[c for c in df.columns if c.startswith("_")])
clean = clean.sort_values("name").reset_index(drop=True)
clean.to_csv(OUT / "countries_clean.csv", index=False, encoding="utf-8")
pd.DataFrame(log).to_csv(OUT / "cleaning_log.csv", index=False, encoding="utf-8")
print(f"raw rows: {n_raw}  clean rows: {len(clean)}  columns: {clean.shape[1]}")
print(pd.DataFrame(log).to_string(index=False))
display(df)


raw rows: 254  clean rows: 254  columns: 23
                                     rule  rows_affected                                                                      note
                      flatten nested JSON            254          languages/currencies/borders -> comma-separated strings + counts
                        '' -> NaN in iso3              4                                                                          
                   '' -> NaN in languages            254                                                                          
              '' -> NaN in language_codes              1                                                                          
                  '' -> NaN in currencies              3                                                                          
              '' -> NaN in currency_codes            254                                                                          
             drop duplicate country nam

,name,iso3,region,population,area_km2,languages,language_codes,currencies,currency_codes,borders,...,is_antarctic_region,num_borders,num_languages,num_currencies,has_sign_or_family_code,population_density,log10_population,log10_area,is_population_outlier,is_area_km2_outlier
0,Abkhazia,<NA>,Asia,244236,8665.0,<NA>,"ab, ru",Russian ruble,<NA>,"GEO, RUS",...,False,2,2,1,False,28.186497,5.387810,3.937769,False,False
1,Afghanistan,AFG,Asia,43844000,652230.0,<NA>,"prs, ps, tk",Afghan afghani,<NA>,"IRN, PAK, TKM, UZB, TJK, CHN",...,False,6,3,1,False,67.221686,7.641910,5.814401,False,False
2,Albania,ALB,Europe,2335930,28748.0,<NA>,sq,Albanian lek,<NA>,"MNE, GRC, MKD, UNK",...,False,4,1,1,False,81.255392,6.368460,4.458608,False,False
3,Algeria,DZA,Africa,47400000,2381741.0,<NA>,ar,Algerian dinar,<NA>,"TUN, LBY, NER, ESH, MRT, MLI, MAR",...,False,7,1,1,False,19.901408,7.675778,6.376895,False,False
4,American Samoa,ASM,Oceania,43268,199.0,<NA>,"en, sm",United States dollar,<NA>,,...,False,0,2,1,False,217.427136,4.636167,2.298853,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,Western Sahara,ESH,Africa,510713,266000.0,<NA>,"mey, es","Algerian dinar, Moroccan dirham, Mauritanian o...",<NA>,"DZA, MRT, MAR",...,False,3,2,3,False,1.919974,5.708177,5.424882,False,False
250,Yemen,YEM,Asia,32684503,527968.0,<NA>,ar,Yemeni rial,<NA>,"OMN, SAU",...,False,2,1,1,False,61.906220,7.514342,5.722608,False,False
251,Zambia,ZMB,Africa,23003214,752612.0,<NA>,en,Zambian kwacha,<NA>,"AGO, BWA, COD, MWI, MOZ, NAM, TZA, ZWE",...,False,8,1,1,False,30.564506,7.361789,5.876571,False,False
252,Zimbabwe,ZWE,Africa,16175966,390757.0,<NA>,"bwg, en, kck, khi, ndc, nd, ny, sn, st, toi, t...","United States dollar, Zimbabwe Gold",<NA>,"BWA, MOZ, ZAF, ZMB",...,False,4,15,2,True,41.396484,7.208870,5.591907,False,False
